# 01A: Exploring the Data Lake

Generic tooling to inspect the CICCADA data lake at three levels: **Storage** (S3 files), **Catalog** (Glue tables + schema), **Data** (actual rows). Reusable for any new data source.

**Three zoom levels:**

| Level | Question | Tool | Cost |
|---|---|---|---|
| **Storage** | What files physically exist? | `s3_ls()` (boto3) | free |
| **Catalog** | What's registered as a queryable table? | `databases()`, `tables()`, `describe()` (Glue) | free |
| **Data** | What do the values look like? | `aq()` (Athena) or `dread()` (DuckDB) | small / local |

Note: If anything says *"token expired"*, run `aws sso login --profile ciccada` in a terminal and re-run the cell.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
sys.path.insert(0, str(pathlib.Path('lib').resolve()))

from shared.aws_config import *          # aq, dread, s3_ls, databases, tables
from shared.ciccada_config import SA, SAI, TABLES
import pandas as pd

## [Level 1] Storage: the physical files in S3

In [2]:
s3_ls()

TokenRetrievalError: Error when retrieving token from sso: Token has expired and refresh failed

In [ ]:
s3_ls('Trino-Warehouse/solar_analytics/')

,type,name,size_mb
0,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
1,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
2,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
3,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
4,folder,Trino-Warehouse/solar_analytics/all_uncurtaile...,None
...,...,...,...
69,folder,Trino-Warehouse/solar_analytics/test_sola_2025...,None
70,folder,Trino-Warehouse/solar_analytics/ts-957d79213e8...,None
71,folder,Trino-Warehouse/solar_analytics/voltwatt_uncur...,None
72,folder,Trino-Warehouse/solar_analytics/voltwatt_uncur...,None


## [Level 2] Catalog: what's queryable, and its schema

Glue is the index that lets us write SQL. These calls are (metadata only).

In [10]:
databases()

,Database,Description
0,bom_nci,
1,default,Default Hive database
2,elb_logdb,
3,sapn2022,
4,solar_analytics,Migrated from Hive Metastore
5,solar_analytics_iceberg,
6,test_db,
7,type_probe,


In [11]:
# Every table in every database, in one view.
all_tabs = pd.concat([tables(d) for d in databases()['Database']], ignore_index=True)
all_tabs

,Database,Table,Description,TableType,Columns,Partitions
0,bom_nci,solar,,EXTERNAL_TABLE,"time, latitude, longitude, surface_global_irra...",
1,elb_logdb,elb_logs_tbl,ALB Access log table,EXTERNAL_TABLE,"type, time, elb, client_ip, client_port, targe...","year, month, day"
2,sapn2022,circuit_measurements,,EXTERNAL_TABLE,"c_id, utc_tstamp, energy, power, voltage, vmin...",
3,sapn2022,circuit_measurements_curtailment_train,,EXTERNAL_TABLE,"c_id, utc_tstamp, energy, power, voltage, vmin...",
4,solar_analytics,circuits,,EXTERNAL_TABLE,"site_id, device_id, circuit_id, device_type, c...",
5,solar_analytics,compliance_voltvar,,EXTERNAL_TABLE,"site_id, s_id, year, month, day, day_night, no...",
6,solar_analytics,compliance_voltwatt,,EXTERNAL_TABLE,"site_id, s_id, year, month, day, noncompliance...",
7,solar_analytics,meta_single_inverters,,EXTERNAL_TABLE,"circuit_id, site_id, device_id, device_type, c...",
8,solar_analytics,meta_single_inverters_wrong_capacity,,EXTERNAL_TABLE,"circuit_id, site_id, device_id, device_type, c...",
9,solar_analytics,meta_single_inverters_wrong_capacity_up2_3c,,EXTERNAL_TABLE,"circuit_id, site_id, device_id, device_type, c...",


In [12]:
# Peek at any table's schema (works on Iceberg via SELECT * LIMIT 1)
table = TABLES['conformance_voltvar']   # 'conformance_voltvar_v2'
# or directly
table = "meta_up23c"
table = "ts"
table_df = aq(f'SELECT * FROM {table} LIMIT 5', database=SAI)

In [13]:
table_df.columns

Index(['circuit_id', 't_stamp', 'power', 'energy', 'energy_reactive',
       'energy_import', 'energy_export', 'energy_reactive_import',
       'energy_reactive_export', 'power_factor', 'voltage', 'current', 'year',
       'month', 'is_pv', 'postcode'],
      dtype='str')

In [14]:
table = "solar"  
table_df = aq(f'SELECT * FROM {table} LIMIT 5', database="bom_nci")

In [15]:
table_df

,time,latitude,longitude,surface_global_irradiance,direct_normal_irradiance,surface_diffuse_irradiance,quality_mask,cloud_type,cloud_optical_depth,solar_elevation,solar_azimuth,postcode,year,month
0,2025-07-01 04:50:00,-43.16,147.78,199.82,612.73,55.95,1,0,0.0,13.6,321.2,7183,2025,7
1,2025-07-01 06:10:00,-43.16,147.78,8.87,0.00,8.87,1,8,0.7,3.0,306.0,7183,2025,7
2,2025-07-01 22:20:00,-43.16,147.78,31.08,0.00,31.08,1,6,19.3,6.1,50.2,7183,2025,7
3,2025-07-01 05:00:00,-43.16,147.78,178.81,584.19,53.23,1,0,0.0,12.4,319.2,7183,2025,7
4,2025-07-01 05:40:00,-43.16,147.78,52.14,18.80,49.76,1,8,0.7,7.3,311.4,7183,2025,7


## [Level 3] Data peek at actual rows

Two engines, same files. 

1. Use `aq()` (Athena) for normal SQL
2. use `dread()` (DuckDB, reads the Parquet file directly) for quick peeks or when Glue's description is wrong.

**Cost:** on the big `ts` table, always filter on `year`/`month`/`is_pv` and never sort on a fresh peek.

In [8]:
aq("SELECT * FROM ts WHERE is_pv = True AND year = 2024 AND month = 1 LIMIT 5", database=SAI)

,circuit_id,t_stamp,power,energy,energy_reactive,energy_import,energy_export,energy_reactive_import,energy_reactive_export,power_factor,voltage,current,year,month,is_pv,postcode
0,483249,2024-01-06 12:30:00,-1.2800,-0.1067,11.1700,0.0,0.1067,11.1700,0.0,0.000091,237.75,0.5745,2024,1,True,2570
1,483249,2024-01-06 12:45:00,-1.4400,-0.1200,11.2308,0.0,0.1200,11.2308,0.0,0.000114,238.55,0.5755,2024,1,True,2570
2,483249,2024-01-06 13:00:00,-1.5067,-0.1256,11.1953,0.0,0.1256,11.1953,0.0,0.000126,238.15,0.5745,2024,1,True,2570
3,483249,2024-01-06 13:05:00,-1.5300,-0.1275,11.1797,0.0,0.1275,11.1797,0.0,0.000130,238.10,0.5740,2024,1,True,2570
4,483249,2024-01-06 14:00:00,-1.4167,-0.1181,11.2528,0.0,0.1181,11.2528,0.0,0.000110,238.85,0.5760,2024,1,True,2570


In [9]:
pd.set_option('display.max_columns', None)
aq("SELECT * FROM structured_data_v2 WHERE year = 2024 AND month = 1 LIMIT 5", database=SAI)

,site_id,t_stamp,actual_day,actual_tod,v,q_kvar_norm,p_kw_norm,s_norm,ghi,cloud_type,cs_day,cs_tod,p_kw_norm_cs,ghi_cs,cloud_type_cs,s_99,ac_capacity_kw,normalization_capacity,normalization_basis,voltage_aggregation,flex_selection,year,month
0,2075357519,2024-01-13 01:30:00,2024-01-13,11:30:00.000,239.650000,-0.003896,0.374581,0.374602,584.92,6,2024-01-29,11:30:00.000,0.917840,997.42,0,8.066811,8.20,8.066811,s_99,avg,exclude,2024,1
1,2075357519,2024-01-15 22:40:00,2024-01-16,08:40:00.000,240.000000,-0.026983,0.602307,0.602911,659.54,0,2024-01-29,08:40:00.000,0.585135,619.00,0,8.066811,8.20,8.066811,s_99,avg,exclude,2024,1
2,1922500063,2024-01-27 22:55:00,2024-01-28,08:55:00.000,243.183333,0.010038,0.060279,0.061109,231.75,7,2024-01-25,08:55:00.000,0.630366,722.27,6,15.804231,15.00,15.804231,s_99,avg,exclude,2024,1
3,725849743,2024-01-22 01:10:00,2024-01-22,11:10:00.000,244.433333,0.000421,0.476413,0.476413,804.50,5,2024-01-21,11:10:00.000,0.855078,1038.66,0,91.593704,110.00,91.593704,s_99,avg,exclude,2024,1
4,1044946999,2024-01-14 20:15:00,2024-01-15,06:15:00.000,245.450000,0.014778,0.243552,0.244000,193.08,0,2024-01-13,06:15:00.000,0.287725,237.56,0,4.970141,4.99,4.970141,s_99,avg,exclude,2024,1


## If Athena chokes: read the Parquet directly with DuckDB

Some results tables have *schema drift*. The Glue description disagrees with the file (e.g. a column the file stores as integer but Glue calls a double). This is common.

Athena won't be able to read it, so DuckDB reads the file's own schema and just works.

The S3 path comes from the error message, or from `s3_ls()`.

In [ ]:
# dread('s3://project-ciccada/Trino-Warehouse/solar_analytics/conformance_voltvar_v2/data/*.parquet')

## Recipe to explore any new data source in future

1. **`s3_ls('<prefix>/')`**: See what physically exists and how it's "foldered".
2. **`databases()` / `tables(db)`**: Check if it is registered in Glue. If yes, can use SQL.
3. **`describe('<table>', db)`**: Learn its columns.
4. **`aq('SELECT ... LIMIT 5')`**: Peek at values (filter on partitions if it's big).
5. Note: Not in Glue, or Glue is wrong?: Point **`dread('s3://.../*.parquet')`** straight at the files.